# 09 - NN vs LR head

Mirrors `notebooks_withseed/08_head_comparison`. Both heads sit on the same fine-tuned backbone, so
this isolates whether the readout choice matters.

In [ ]:
import os, sys, json
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import warnings; warnings.filterwarnings('ignore')
import numpy as np, torch
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
OSF, OUT = '../data/preprocessed', '../results_parcel'
MODEL, SUFFIX = 'tiny', 'cortical'
cv = StratifiedKFold(5, shuffle=True, random_state=123)

ft = f'{OUT}/finetune_combined_{MODEL}.pt'
assert os.path.exists(ft), 'run notebook 04 first'
d = torch.load(ft, map_location='cpu', weights_only=False)
cvr, y_all = d['cv'], d['labels']
nfs = sorted(cvr.keys(), reverse=True)
FROZEN_REF, HANDCRAFT_REF = 0.831, 0.872

nn_v = [cvr[nf]['nn_mean'] for nf in nfs]; nn_e = [np.std(cvr[nf]['nn_folds']) for nf in nfs]
lr_v = [cvr[nf]['lr_mean'] for nf in nfs]; lr_e = [np.std(cvr[nf]['lr_folds']) for nf in nfs]

x, w = np.arange(len(nfs)), 0.34
fig, ax = plt.subplots(figsize=(max(7, 3*len(nfs)), 6))
b1 = ax.bar(x - w/2, nn_v, w, yerr=nn_e, capsize=5, color='#B07AA1',
            error_kw=dict(lw=1.2, ecolor='#444'), label='NN head (5-fold CV)')
b2 = ax.bar(x + w/2, lr_v, w, yerr=lr_e, capsize=5, color='#59A14F',
            error_kw=dict(lw=1.2, ecolor='#444'), label='LR head (5-fold CV)')
ax.bar_label(b1, fmt='%.3f', padding=12, fontsize=9); ax.bar_label(b2, fmt='%.3f', padding=12, fontsize=9)
ax.axhline(FROZEN_REF,    color='green', ls='--', lw=1.3, label='Frozen + LR, sensor-level (0.831)')
ax.axhline(HANDCRAFT_REF, color='red',   ls=':',  lw=1.3, label='Handcrafted + LR, sensor-level (0.872)')
ax.set_xticks(x); ax.set_xticklabels([f'Last {11-nf} block(s)' for nf in nfs])
ax.set_ylabel('Accuracy'); ax.set_ylim(0.4, 1.0)
ax.set_title(f'NN vs LR Head - Combined cohort, {len(y_all)} subjects (BrainOmni-{MODEL})')
ax.legend(fontsize=9, ncol=2); ax.grid(axis='y', alpha=0.3); ax.set_axisbelow(True)
plt.tight_layout(); plt.savefig(f'{OUT}/parcel_head_nn_vs_lr_{MODEL}.png', dpi=150); plt.show()

print(f'{"config":16s} {"NN":>8} {"LR":>8}')
for nf, a, b in zip(nfs, nn_v, lr_v):
    print(f'last-{11-nf:<11d} {a:8.3f} {b:8.3f}')